# 韩国半导体出口英文档案全月来源资格

本卡仅核 2019-01 至 2026-09 的 93 个月官方档案覆盖、现有 PDF 元数据与登记日差距、可恢复 NAS 原字节和 Qlib 交易日映射。不提取收益标签，不拟合或运行账户。

In [ ]:
from pathlib import Path
import os,json,hashlib,datetime
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/korea_semiconductor_exports_full_source_20261022"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized"]
assert not (card/"run_state.json").exists(),"one fixed source qualification attempt already started"
for path,digest in plan["source_identities_sha256"].items():assert sha(root/path)==digest,path
source=root/plan["source_card"]
manifest=json.loads((source/"source_snapshot_manifest.json").read_text())
binding=json.loads((source/"nas_source_archive_binding.json").read_text())
assert len(manifest["entries"])==manifest["files"]==binding["source_file_count"]==446
assert binding["NAS_remote_SHA256_verified"] and binding["source_manifest_sha256"]==sha(source/"source_snapshot_manifest.json")
assert sha(root/plan["calendar_provider"]/"qlib_store_manifest.json")==plan["calendar_manifest_sha256"]
assert sha(root/plan["calendar_provider"]/"calendars/day.txt")==plan["calendar_file_sha256"]
articles=json.loads((source/"article_html_audit.json").read_text())
pdfs=json.loads((source/"english_pdf_coverage_audit.json").read_text())
candidates=json.loads((source/"national_release_candidate_manifest_v3.json").read_text())
assert articles["months"]==pdfs["months"]==candidates["target_months"]==93
assert candidates["status"]=="one_national_release_candidate_each_month"
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(root/plan["calendar_provider"]),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2019-01-01",end_time="2026-10-21",freq="day")).normalize()
assert len(calendar)>1000 and calendar.is_monotonic_increasing
rows=[]
for month in sorted(articles["month_details"]):
    article=articles["month_details"][month];pdf=pdfs["per_month"][month]
    assert len(article["PDF_links"])==1 and len(candidates["articles_by_month"][month])==1
    reg=pd.Timestamp(article["registration_date"])
    raw=pdf["raw_CreationDate"]
    assert raw and raw.startswith("D:")
    created=pd.Timestamp(datetime.datetime.strptime(raw[2:10],"%Y%m%d"))
    following_registration=calendar[calendar>reg]
    following_metadata=calendar[calendar>created]
    rows.append({"observation_month":month,"article_id":article["article_id"],
                 "English_registration_date":str(reg.date()),"current_PDF_creation_date":str(created.date()),
                 "PDF_creation_lag_calendar_days":pdf["creation_lag_calendar_days_after_English_registration"],
                 "first_US_session_after_registration":str(following_registration[0].date()) if len(following_registration) else None,
                 "first_US_session_after_current_PDF_creation_metadata":str(following_metadata[0].date()) if len(following_metadata) else None,
                 "HTML_semiconductor_word_present":article["has_semiconductor_category_text"],
                 "PDF_semiconductor_word_present":pdf["semiconductor_word_count"]>0,
                 "PDF_SHA256":pdf["PDF_SHA256"]})
table=pd.DataFrame(rows)
assert len(table)==93 and table.observation_month.iloc[0]=="2019-01" and table.observation_month.iloc[-1]=="2026-09"
late7=int((table.PDF_creation_lag_calendar_days>7).sum())
late30=int((table.PDF_creation_lag_calendar_days>30).sum())
assert late7==pdfs["creation_lag_gt_7d"]==69 and late30==pdfs["creation_lag_gt_30d"]==26
result={"status":"complete_93_month_English_source_archive_historical_vintage_not_qualified",
        "months":len(table),"first_month":table.observation_month.iloc[0],"last_month":table.observation_month.iloc[-1],
        "original_source_files_archived":binding["source_file_count"],"source_archive_sha256":binding["archive_sha256"],
        "source_archive_remote_hash_verified":True,"PDF_creation_lag_gt_7d":late7,"PDF_creation_lag_gt_30d":late30,
        "English_HTML_no_semiconductor_months":articles["months_without_semiconductor_word"],
        "English_PDF_no_semiconductor_months":pdfs["no_semiconductor_text_months"],
        "Korean_two_sample_vintage_audit":"research/korea_semiconductor_exports_source_20261022/korean_vintage_result.json",
        "historical_first_public_vintage_qualified":False,"numeric_monthly_factor_values_qualified":False,
        "market_outcome_reads":0,"D_calendar_calls":1,"D_features_calls":0,"model_fits":0,"accounts":0,
        "boundary":"The current English PDF file dates are often much later than article registration and cannot be treated as original publication vintages. Qlib session mappings are descriptive only, not tradeable factor rows."}
(card/"run_state.json").write_text(json.dumps({"status":"started","plan_sha256":sha(plan_path)})+chr(10))
tracking=card/"qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="sources",source_plan=plan,archive_manifest=manifest,
                     NAS_binding=binding,article_audit=articles,PDF_audit=pdfs)
    rec.save_objects(artifact_path="analysis",source_qualification=result,monthly_source_clocks=table)
    rec.log_metrics(source_months=len(table),source_PDF_late_gt_7d=late7,
                    source_PDF_late_gt_30d=late30,first_public_qualified=0,model_fits=0,accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),
                  recorder_uri=str(tracking),recorder_local_dir=str(tracking/str(rec.experiment_id)/rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(p.relative_to(card)),
    "sha256":sha(p),"bytes":p.stat().st_size} for p in sorted(artifacts.rglob("*")) if p.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(verification,indent=2)+chr(10))
(card/"source_qualification_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":plan["experiment"],"status":result["status"],
         "decision":"source_coverage_complete_original_vintage_needed_before_predictive_screen",
         "result":"source_qualification_result.json","recorder_id":result["recorder_id"],
         "experiment_id":result["experiment_id"],"recorder_uri":result["recorder_uri"],
         "recorder_local_dir":result["recorder_local_dir"],"model_fits":0,"accounts":0,
         "qualified_forward_days":0,"objective_achieved":False,"artifact_policy":plan["artifact_policy"]}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card/"conclusion.md").write_text("# 中文结论\n\n韩国主管部门英文档案从 2019-01 至 2026-09 逐月唯一对应 "+str(len(table))+" 篇全国出口公告，每月都有当前 HTML 与 PDF；446 份原始来源文件在 NAS 保留可恢复字节，归档哈希经远端读回验证。当前英文 PDF 有 "+str(late7)+" 份创建时间晚于网页登记日超过一周，"+str(late30)+" 份超过一月；2022-03 PDF 没有可提取的半导体字样。Recorder `"+result["recorder_id"]+"` 保存逐月来源时钟与缺口。\n\n这些材料证明目录覆盖和数值来源潜力，不证明当年首发版本。用 PDF 创建日期或今天下载的 HTML 回填 2019–2026 将引入时序不确定性；本卡没有读 SOXX 收益、拟合或交易账户。下一步若继续韩国出口机制，先取得可证明当期公开的韩文公告版本或开展未来逐月原字节捕获，之后才冻结信息检验。\n")
(card/"run_state.json").write_text(json.dumps({"status":"completed","recorder_id":result["recorder_id"],"plan_sha256":sha(plan_path)})+chr(10))
print(json.dumps({"status":result["status"],"months":len(table),"late7":late7,"late30":late30,"recorder_id":result["recorder_id"]}))

## 中文结论

完整 93 月来源覆盖不等于历史 PIT 合格；当前英文 PDF 多数晚于登记日，详情与 Recorder 身份见 `conclusion.md`、`source_qualification_result.json`。